# ⚡ Topic 02: Apache Spark Architecture & Resilient Distributed Datasets (RDDs)

## 1. Apache Spark Architecture

```
                       ┌─────────────────────────┐
                       │     Driver Program      │
                       │   (SparkSession / DAG)  │
                       └────────────┬────────────┘
                                    │
                                    ▼
                       ┌─────────────────────────┐
                       │     Cluster Manager     │
                       │(YARN / K8s / Standalone)│
                       └────────────┬────────────┘
                                    │
           ┌────────────────────────┴────────────────────────┐
           ▼                                                 ▼
┌─────────────────────┐                           ┌─────────────────────┐
│    Worker Node      │                           │    Worker Node      │
│ ┌─────────────────┐ │                           │ ┌─────────────────┐ │
│ │  Executor Task  │ │                           │ │  Executor Task  │ │
│ └─────────────────┘ │                           │ └─────────────────┘ │
└─────────────────────┘                           └─────────────────────┘
```

---

## 2. Resilient Distributed Datasets (RDDs)
An **RDD** is an immutable, partitioned collection of records operating across cluster nodes.
- **Resilient:** Recomputes missing/failed partitions automatically using lineage graph.
- **Distributed:** Split across worker nodes for parallel execution.
- **Lazy Evaluation:** Transformations build a DAG (Lineage Graph); execution occurs ONLY when an Action is called.

---

## 3. Hands-on: PySpark RDD Operations & Lineage Inspection


In [ ]:
from pyspark.sql import SparkSession

# Initialize local SparkSession
spark = SparkSession.builder \
    .appName("MLOps_Spark_Architecture_Demo") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("⚡ SparkSession Initialized!")
print(f"   Spark Version: {spark.version}")
print(f"   App Name:      {spark.sparkContext.appName}")

# Create an RDD from a Python list
numbers = list(range(1, 101))
rdd_numbers = sc.parallelize(numbers, numSlices=4)

# Apply Transformations (Lazy)
rdd_filtered = rdd_numbers.filter(lambda x: x % 2 == 0)
rdd_mapped = rdd_filtered.map(lambda x: (x, x ** 2))

# Inspect Lineage Graph (DAG)
print("\n🔍 RDD Lineage Graph (.toDebugString()):")
print(rdd_mapped.toDebugString().decode('utf-8'))

# Trigger Action (Executes computation across partitions)
results = rdd_mapped.take(5)
total_sum = rdd_mapped.map(lambda x: x[1]).reduce(lambda a, b: a + b)

print("\n📊 Action Results:")
print("   First 5 (Even Number, Square):", results)
print("   Sum of Squares of Evens:", total_sum)
